In [37]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
# from langchain.retrievers import EnsembleRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
# from langchain.retrievers.self_query.base import SelfQueryRetriever
# from langchain.chains.query_constructor.base import AttributeInfo

In [34]:
import langchain
print(langchain.__version__)

1.2.0


In [30]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [31]:
vectorstore = Chroma(
    persist_directory="chroma_gempa_db",
    embedding_function=embeddings
)

In [32]:
# Load database yang sudah ada
db = Chroma(persist_directory="./chroma_gempa_db", embedding_function=embeddings)

# Uji coba pertanyaan
query = "Bagaimana aktivitas gempa di wilayah jambi?"
docs = db.similarity_search(query, k=2)

for doc in docs:
    print(f"Hasil relevan: {doc.page_content}")

Hasil relevan: Pada tanggal 24 June 1990 pukul 00:00 WIB terjadi gempa bumi di Provinsi Gorontalo dengan magnitudo 5.1 dan kedalaman 153.3 km. Gempa ini termasuk gempa menengah dan berlokasi pada koordinat lintang 0.483 dan bujur 122.389.
Hasil relevan: Pada tanggal 17 June 1997 pukul 00:00 WIB terjadi gempa bumi di Provinsi Nusa Tenggara Timur dengan magnitudo 5.2 dan kedalaman 145.3 km. Gempa ini termasuk gempa menengah dan berlokasi pada koordinat lintang -8.222 dan bujur 122.78.


In [15]:
query = "gempa di Jawa Barat"
docs = vectorstore.similarity_search(query, k=5)

for i, d in enumerate(docs, 1):
    print(f"Dokumen {i}")
    print(d.page_content)
    print(d.metadata)
    print("----")

Dokumen 1
Pada tanggal 20 November 2016 pukul 00:00 WIB terjadi gempa bumi di Provinsi Nusa Tenggara Barat dengan magnitudo 2.1 dan kedalaman 10.0 km. Gempa ini termasuk gempa dangkal dan berlokasi pada koordinat lintang -8.23 dan bujur 118.11.
{'year': 2016, 'date': '11/20/2016', 'month': 11, 'depth_category': 'dangkal', 'magnitude': 2.1, 'province': 'Nusa Tenggara Barat'}
----
Dokumen 2
Pada tanggal 22 September 2016 pukul 00:00 WIB terjadi gempa bumi di Provinsi Nusa Tenggara Barat dengan magnitudo 4.1 dan kedalaman 10.0 km. Gempa ini termasuk gempa dangkal dan berlokasi pada koordinat lintang -8.17 dan bujur 118.69.
{'magnitude': 4.1, 'province': 'Nusa Tenggara Barat', 'month': 9, 'depth_category': 'dangkal', 'date': '9/22/2016', 'year': 2016}
----
Dokumen 3
Pada tanggal 09 June 2016 pukul 00:00 WIB terjadi gempa bumi di Provinsi Nusa Tenggara Barat dengan magnitudo 3.0 dan kedalaman 10.0 km. Gempa ini termasuk gempa dangkal dan berlokasi pada koordinat lintang -8.06 dan bujur 118.

In [28]:
query = "Apakah ada gempa pada bulan march di Aceh?"
docs = vectorstore.similarity_search(query, k=5)

for d in docs:
    print(d.page_content)

Pada tanggal 31 March 2016 pukul 00:00 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 5.1 dan kedalaman 11.08 km. Gempa ini termasuk gempa dangkal dan berlokasi pada koordinat lintang 3.1211 dan bujur 96.2001.
Pada tanggal 20 June 2015 pukul 00:00 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 1.9 dan kedalaman 12.0 km. Gempa ini termasuk gempa dangkal dan berlokasi pada koordinat lintang 3.45 dan bujur 97.81.
Pada tanggal 20 August 2016 pukul 00:00 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 3.6 dan kedalaman 0.0 km. Gempa ini termasuk gempa dangkal dan berlokasi pada koordinat lintang 3.3395 dan bujur 96.1227.
Pada tanggal 10 February 1981 pukul 00:00 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 5.1 dan kedalaman 121.0 km. Gempa ini termasuk gempa menengah dan berlokasi pada koordinat lintang 3.985 dan bujur 97.227.
Pada tanggal 17 August 2016 pukul 00:00 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 3.8 dan kedalaman 12.0 km. G

In [6]:
docs = vectorstore.similarity_search(
    "gempa bulan agustus",
    k=5,
    filter={"place": "Jawa Timur"}
)
for d in docs:
    print(d.page_content)

groq

In [17]:
from langchain_groq import ChatGroq
import os

# Set API Key Groq
os.environ["GROQ_API_KEY"] = "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"

# Ganti llama3-8b-8192 menjadi llama-3.1-8b-instant
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

In [16]:
from langchain_community.query_constructors.chroma import ChromaTranslator
# from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

# Jika masih error di baris pertama, pastikan langchain sudah terupdate:
# pip install --upgrade langchain
# 1. Definisikan informasi kolom metadata kamu
metadata_field_info = [
    AttributeInfo(name="province", description="Nama provinsi lokasi gempa", type="string"),
    AttributeInfo(name="year", description="Tahun terjadinya gempa", type="integer"),
    AttributeInfo(name="depth_category", description="Kategori kedalaman gempa (dangkal, menengah, dalam)", type="string"),
    AttributeInfo(name="magnitude", description="Kekuatan gempa dalam skala Richter", type="float"),
]

document_content_description = "Ringkasan kejadian gempa bumi di Indonesia"

# 2. Inisialisasi Self-Query Retriever
# Menggunakan Groq sebagai 'asisten' untuk menerjemahkan pertanyaan user menjadi filter
retriever = SelfQueryRetriever.from_llm(
    llm, 
    vectorstore, 
    document_content_description, 
    metadata_field_info, 
    verbose=True
)

ModuleNotFoundError: No module named 'langchain.chains'

In [18]:
import json

def get_metadata_filters(query):
    system_instruction = """
    Tugas Anda adalah mengekstrak filter pencarian dari pertanyaan user terkait gempa.
    Output harus berupa JSON murni tanpa penjelasan apapun.
    Field yang tersedia: province (string), year (integer), magnitude (float).
    
    Contoh:
    User: "Gempa di Sulawesi Selatan tahun 2024"
    Output: {"province": "Sulawesi Selatan", "year": 2024}
    """
    
    # Memanggil Groq
    res = llm.invoke([("system", system_instruction), ("human", query)])
    
    try:
        # Membersihkan output jika ada karakter aneh
        clean_res = res.content.strip().replace('```json', '').replace('```', '')
        return json.loads(clean_res)
    except:
        return {}

In [19]:
# 1. Ambil filter dari pertanyaan user
user_query = "Cari gempa di Sulawesi Selatan yang terjadi tahun 2024"
filters = get_metadata_filters(user_query)

# 2. Cari di Chroma dengan filter tersebut
# Jika filter kosong, dia akan melakukan pencarian biasa
docs = vectorstore.similarity_search(
    user_query, 
    k=3, 
    filter=filters if filters else None
)

# 3. Gabungkan hasil ke LLM (RAG)
context = "\n".join([doc.page_content for doc in docs])

final_prompt = f"""
Gunakan data berikut untuk menjawab pertanyaan: {user_query}
Data: {context}
"""

print(llm.invoke(final_prompt).content)

ValueError: Expected where to have exactly one operator, got {'province': 'Sulawesi Selatan', 'year': 2024} in query.

In [1]:
import json
import re

def get_metadata_filters(query):
    system_instruction = """
    Tugas Anda mengekstrak filter dari pertanyaan user ke dalam JSON.
    HANYA keluarkan JSON.
    
    ATURAN CHROMA DB:
    1. Jika hanya 1 filter: {"key": "value"}
    2. Jika lebih dari 1 filter, gunakan $and: {"$and": [{"key1": "val1"}, {"key2": "val2"}]}
    
    Field: "province" (string), "year" (int), "magnitude" (float).
    """
    
    res = llm.invoke([("system", system_instruction), ("human", query)])
    
    try:
        # Ekstrak JSON menggunakan regex agar lebih aman
        import re
        json_str = re.search(r'\{.*\}', res.content, re.DOTALL).group()
        filters = json.loads(json_str)
        
        # Validasi manual: Jika ada > 1 key tapi belum pakai $and, kita bungkus otomatis
        if len(filters) > 1 and "$and" not in filters:
            new_filters = {"$and": []}
            for k, v in filters.items():
                new_filters["$and"].append({k: v})
            return new_filters
            
        return filters
    except:
        return None

def tanya_gempa(user_query):
    # 1. Ekstrak Filter Metadata
    filters = get_metadata_filters(user_query)
    print(f"DEBUG - Filter yang digunakan: {filters}")
    
    # 2. Cari di ChromaDB
    # Kita gunakan similarity search dengan filter metadata
    docs = vectorstore.similarity_search(user_query, k=3, filter=filters if filters else None)
    
    # 3. Gabungkan Konteks
    context = "\n\n".join([d.page_content for d in docs])
    
    # 4. Prompt Engineering Akhir
    prompt_analis = f"""
    Anda adalah Analis Gempa BMKG Virtual. Berdasarkan data di bawah ini, berikan laporan singkat.

    DATA KONTEKS:
    {context}

    PERTANYAAN: {user_query}

    INSTRUKSI:
    1. Jika ada lebih dari satu gempa, sajikan dalam bentuk TABEL Markdown.
    2. Kolom tabel: No, Tanggal, Lokasi, Magnitudo, Kedalaman.
    3. Berikan satu kalimat kesimpulan di akhir tentang tren gempa tersebut.
    """
    
    return llm.invoke(prompt_analis).content

# Coba jalankan
print(tanya_gempa("gempa terdangkal"))

NameError: name 'llm' is not defined

In [8]:
from groq import Groq

client = Groq(api_key="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")

prompt = "jelaskan gempa terbesar 2024"

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",     # pilih model
    messages=[
        {"role": "user", "content": "halo"}
    ],
    max_tokens=256,
    temperature=0.2
)

print(response.choices[0].message.content)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Halo! How can I assist you today?


In [10]:
from groq import Groq

client = Groq(api_key="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")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "halo"}],
    max_tokens=256,
    temperature=0.2
)

print("Response:", response.choices[0].message.content)
print("Usage:", response.usage)


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Response: Halo! How can I assist you today?
Usage: CompletionUsage(completion_tokens=11, prompt_tokens=37, total_tokens=48, completion_time=0.03133477, completion_tokens_details=None, prompt_time=0.001102308, prompt_tokens_details=None, queue_time=0.053556516, total_time=0.032437078)


In [ ]:
print(response.usage)

CompletionUsage(completion_tokens=11, prompt_tokens=37, total_tokens=48, completion_time=0.03133477, completion_tokens_details=None, prompt_time=0.001102308, prompt_tokens_details=None, queue_time=0.053556516, total_time=0.032437078)


In [9]:
print("Model dipakai:", response.model)
print("Output:", response.choices[0].message.content)


Model dipakai: llama-3.3-70b-versatile
Output: Mohon maaf, saya tidak memiliki informasi tentang gempa terbesar pada tahun 2024 karena saya hanya memiliki data hingga tahun 2023. Namun, saya dapat memberikan informasi umum tentang gempa bumi dan bagaimana gempa besar dapat terjadi.

Gempa bumi adalah getaran atau guncangan yang terjadi di permukaan bumi, biasanya disebabkan oleh pergerakan lempeng tektonik. Gempa bumi dapat menyebabkan kerusakan parah pada bangunan, infrastruktur, dan lingkungan sekitar.

Beberapa faktor yang dapat mempengaruhi kekuatan gempa bumi antara lain:

1. Kedalaman gempa: Gempa bumi yang terjadi pada kedalaman yang lebih dangkal cenderung lebih kuat dan merusak daripada gempa bumi yang terjadi pada kedalaman yang lebih dalam.
2. Magnitudo gempa: Magnitudo gempa bumi adalah ukuran kekuatan gempa bumi. Semakin besar magnitudo, semakin kuat gempa bumi.



In [4]:
from langchain.schema import Document

ModuleNotFoundError: No module named 'langchain.schema'

In [ ]:
import logging
import numpy as np
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document



# Setup Logging agar terlihat pro seperti kode SBM
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class GempaExpert:
    def __init__(self, persist_directory: str):
        # 1. Inisialisasi Embedding Model (Multilingual untuk konteks Indonesia)
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        )
        
        # 2. Load ChromaDB
        self.vector_db = Chroma(
            persist_directory=persist_directory, 
            embedding_function=self.embeddings
        )
        
        # 3. Inisialisasi Re-ranker (Cross-Encoder)
        # Ini yang akan memastikan Aceh tidak tertukar dengan NTT
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        
        # 4. Siapkan BM25 (Keyword Search)
        self._prepare_bm25()
        
        logger.info("GempaExpert dengan Hybrid Search & Re-ranker siap!")

    def _prepare_bm25(self):
        """Mengambil semua data dari Chroma untuk membangun index BM25"""
        all_data = self.vector_db.get()
        if not all_data['documents']:
            raise ValueError("ChromaDB kosong. Pastikan data sudah di-index.")

        # Buat list Document untuk BM25
        self.all_docs = [
            Document(page_content=txt, metadata=meta)
            for txt, meta in zip(all_data['documents'], all_data['metadatas'])
        ]

        # Build BM25 index
        self.bm25 = BM25Retriever.from_documents(self.all_docs)
        self.bm25.k = 10  # Ambil 10 kandidat awal dari keyword

    def answer_question(self, question: str, top_k: int = 100):
        # --- TAHAP 1: HYBRID RETRIEVAL ---
        # Ambil dari Vector (Makna)
        vector_results = self.vector_db.similarity_search(question, k=30)
        # Ambil dari BM25 (Keyword eksak)
        bm25_results = self.bm25.invoke(question)[:30]
        
        # Gabungkan dan hapus duplikat berdasarkan isi teks
        combined = {doc.page_content: doc for doc in (vector_results + bm25_results)}.values()
        candidate_docs = list(combined)
        
        # --- TAHAP 2: RE-RANKING (Logika SBM) ---
        # Menilai ulang kecocokan query dengan setiap kandidat dokumen
        pairs = [[question, doc.page_content] for doc in candidate_docs]
        scores = self.reranker.predict(pairs)
        
        # Masukkan skor ke metadata dan urutkan
        for i, doc in enumerate(candidate_docs):
            doc.metadata['rerank_score'] = scores[i]
        
        # Urutkan dari skor tertinggi (descending)
        reranked_docs = sorted(candidate_docs, key=lambda x: x.metadata['rerank_score'], reverse=True)
        
        # Ambil top_k terbaik
        final_context = reranked_docs[:top_k]
        
        return final_context

# ==========================================
# CARA PAKAI
# ==========================================
# expert = GempaExpert(persist_directory="./chroma_gempa_db")
# results = expert.answer_question("Jelaskan gempa di Jawa Barat")

# for doc in results:
#     print(f"Skor: {doc.metadata['rerank_score']:.4f} | Teks: {doc.page_content[:100]}...")

e:\COOLYEAH\PA\gempa\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# CARA PAKAI
# ==========================================
expert = GempaExpert(persist_directory="./chroma_gempa_db")
results = expert.answer_question("Jelaskan gempa di Aceh 2022")

for doc in results:
    print(f"Skor: {doc.metadata['rerank_score']:.4f} | Teks: {doc.page_content[:100]}...")


C:\Users\calys\AppData\Local\Temp\ipykernel_23156\2138124955.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


C:\Users\calys\AppData\Local\Temp\ipykernel_23156\2138124955.py:23: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_db = Chroma(
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
INFO:__main__:GempaExpert dengan Hybrid Search & Re-ranker siap!
Batches: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]

Skor: 5.8994 | Teks: Pada tanggal 23 August 2022 pukul 16:45 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.6...
Skor: 5.8488 | Teks: Pada tanggal 12 June 2022 pukul 17:45 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.7 d...
Skor: 5.7150 | Teks: Pada tanggal 02 September 2022 pukul 03:34 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo ...
Skor: 5.6712 | Teks: Pada tanggal 15 March 2022 pukul 09:08 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.7 ...
Skor: 5.6596 | Teks: Pada tanggal 22 June 2022 pukul 06:48 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.9 d...
Skor: 5.6526 | Teks: Pada tanggal 11 September 2022 pukul 15:35 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo ...
Skor: 5.6418 | Teks: Pada tanggal 20 June 2022 pukul 05:01 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.6 d...
Skor: 5.5905 | Teks: Pada tanggal 16 August 2022 pukul 07:02 WIB terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.6...


In [2]:
api_key='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'

In [4]:
import time
from groq import Groq

class GempaExpert(GempaExpert): # Melanjutkan kelas yang sudah kamu buat
    def set_groq_client(self, api_key: str):
        self.client = Groq(api_key=api_key)
        self.model_name = "llama-3.3-70b-versatile"

    def generate_answer(self, question: str):
        start_time = time.time()
        
        # 1. Ambil konteks terbaik dari hasil Re-ranker
        context_docs = self.answer_question(question, top_k=100)
        
        # 2. Gabungkan teks dokumen menjadi satu blok konteks
        context_block = "\n\n".join([
            f"Data {i+1}:\n{doc.page_content}" 
            for i, doc in enumerate(context_docs)
        ])
        
        # 3. Prompt Khusus Analisis Gempa
        prompt = f"""
Anda adalah Sistem Pakar Seismologi. Tugas Anda adalah memberikan ringkasan analisis berdasarkan data gempa yang tersedia.

KONTEKS DATA:
{context_block}

PERTANYAAN USER: 
{question}

INSTRUKSI JAWABAN:
- Gunakan Bahasa Indonesia yang formal.
- Jika ada tren (misal: banyak gempa dangkal), sebutkan.
- Sebutkan detail waktu, magnitudo, dan lokasi secara spesifik.
- Jika data tidak relevan, katakan data tidak ditemukan.

JAWABAN:
"""
        
        # 4. Generate via Groq
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2
            )
            answer = response.choices[0].message.content
            duration = time.time() - start_time
            return answer, duration
        except Exception as e:
            return f"Error saat generate jawaban: {e}", 0

# ==========================================
# MENU INTERAKTIF (Adaptasi dari kode SBM)
# ==========================================
def run_interactive(expert):
    print("\n" + "="*40)
    print("   SISTEM ANALISIS DATA GEMPA RAG   ")
    print("="*40)
    
    while True:
        print("\nMenu:")
        print("1. Tanya Data Gempa (Narrative)")
        print("2. Debug Retrieval (Lihat Skor Re-rank)")
        print("3. Keluar")
        
        pilihan = input("\nPilih menu (1-3): ")
        
        if pilihan == "1":
            q = input("Pertanyaan Anda: ")
            print("\nSedang menganalisis...")
            jawaban, durasi = expert.generate_answer(q)
            print("-" * 20)
            print(jawaban)
            print("-" * 20)
            print(f"Selesai dalam {durasi:.2f} detik.")
            
        elif pilihan == "2":
            q = input("Pertanyaan untuk Debug: ")
            results = expert.answer_question(q, top_k=5)
            print(f"\nTop 5 Dokumen Relevan untuk: '{q}'")
            for i, doc in enumerate(results):
                print(f"{i+1}. [Skor: {doc.metadata['rerank_score']:.4f}] {doc.page_content[:150]}...")
                
        elif pilihan == "3":
            print("Keluar... Sampai jumpa!")
            break

In [9]:
if __name__ == "__main__":
    DB_PATH = "./chroma_gempa_db"
    GROQ_KEY = api_key

    try:
        # Inisialisasi database dan retriever
        expert = GempaExpert(DB_PATH)

        # Inisialisasi Groq client
        expert.set_groq_client(api_key)
        
        query = "Bagaimana pola gempa yang terjadi di wilayah Jawa Barat tahun 2022?"
        print(f"\nUser: {query}")
        
        jawaban, durasi = expert.generate_answer(query)
        print(f"\nGempaExpert: {jawaban}")
        print(f"\n(Proses selesai dalam {durasi:.2f} detik)")

    except Exception as e:
        print(f"Error: {e}")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
INFO:__main__:GempaExpert dengan Hybrid Search & Re-ranker siap!



User: Bagaimana pola gempa yang terjadi di wilayah Jawa Barat tahun 2022?


Batches: 100%|██████████| 4/4 [00:05<00:00,  1.41s/it]
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



GempaExpert: Berdasarkan data yang tersedia, pada tahun 2022, terdapat beberapa gempa bumi yang terjadi di wilayah Jawa Barat. Berikut adalah beberapa gempa bumi yang tercatat:

1. Pada tanggal 25 November 2016 (tidak termasuk tahun 2022, data tidak relevan)
2. Pada tanggal 28 July 2007 (tidak termasuk tahun 2022, data tidak relevan)
3. Pada tanggal 17 November 1992 (tidak termasuk tahun 2022, data tidak relevan)
4. Pada tanggal 26 October 1997 (tidak termasuk tahun 2022, data tidak relevan)
5. Pada tanggal 09 April 2014 (tidak termasuk tahun 2022, data tidak relevan)
6. Pada tanggal 29 July 2008 (tidak termasuk tahun 2022, data tidak relevan)
7. Pada tanggal 07 June 1992 (tidak termasuk tahun 2022, data tidak relevan)

Namun, tidak ada data yang mencatat gempa bumi di Jawa Barat pada tahun 2022. Oleh karena itu, dapat disimpulkan bahwa data tidak ditemukan untuk pola gempa di Jawa Barat pada tahun 2022.

(Proses selesai dalam 9.01 detik)


In [4]:
import logging
import time
from typing import List

import numpy as np
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder
from groq import Groq


# ====================================================
# LOGGING
# ====================================================
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("GempaExpert")


# ====================================================
# KELAS UTAMA RAG + LLM
# ====================================================
class GempaExpert:
    def __init__(self, persist_directory: str, groq_api_key: str):
        """Inisialisasi sistem RAG: Embedding + Chroma + BM25 + Reranker + Groq LLM"""

        # 1. Embedding multilingual
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
        )

        # 2. Load ChromaDB
        self.vector_db = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings
        )

        # 3. Cross Encoder Re-ranker
        self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

        # 4. Build BM25 untuk keyword matching
        self._prepare_bm25()

        # 5. Groq Client
        self.client = Groq(api_key=groq_api_key)
        self.model_name = "llama-3.3-70b-versatile"

        logger.info("GempaExpert siap: Hybrid Retrieval + Re-ranker + LLM diaktifkan.")

    # ------------------------------------------------
    # BUILD BM25
    # ------------------------------------------------
    def _prepare_bm25(self):
        """Mengambil semua dokumen untuk BM25"""
        data = self.vector_db.get()

        if not data["documents"]:
            raise ValueError("ChromaDB kosong! Pastikan sudah diisi data.")

        self.all_docs = [
            Document(page_content=text, metadata=meta)
            for text, meta in zip(data["documents"], data["metadatas"])
        ]

        self.bm25 = BM25Retriever.from_documents(self.all_docs)
        self.bm25.k = 100  # ambil kandidat keyword

        logger.info(f"BM25 siap dengan {len(self.all_docs)} dokumen.")

    # ------------------------------------------------
    # HYBRID RETRIEVAL + RERANKING
    # ------------------------------------------------
    def answer_question(self, question: str, top_k: int = 100):
        """Hybrid Search (Vector + BM25) + Re-Ranking"""

        # Vector search
        vect = self.vector_db.similarity_search(question, k=1000)

        # Keyword search
        bm25 = self.bm25.invoke(question)[:1000]

        # Gabungkan dan hapus duplikat
        combined = {doc.page_content: doc for doc in (vect + bm25)}.values()
        candidate_docs = list(combined)

        # Re-ranking pairs
        pairs = [[question, doc.page_content] for doc in candidate_docs]
        scores = self.reranker.predict(pairs)

        # Simpan skor ke metadata
        for i, doc in enumerate(candidate_docs):
            doc.metadata["rerank_score"] = float(scores[i])

        # Urutkan
        ranked = sorted(candidate_docs, key=lambda d: d.metadata["rerank_score"], reverse=True)

        return ranked[:top_k]

    # ------------------------------------------------
    # GENERATE JAWABAN LLM
    # ------------------------------------------------
    def generate_answer(self, question: str):
        """Membangun prompt dan generate jawaban dengan LLM Groq"""

        start = time.time()

        docs = self.answer_question(question, top_k=100)

        context = "\n\n".join(
            f"- Data {i+1}: {doc.page_content}" for i, doc in enumerate(docs)
        )

        prompt = f"""
Anda adalah sistem pakar seismologi.

KONTEKS DATA GEMPA:
{context}

PERTANYAAN:
{question}

INSTRUKSI:
- Jawab berdasarkan data saja, jangan mengarang.
- Sertakan detail waktu, lokasi, magnitudo, kedalaman bila ada.
- Bila data tidak ditemukan, katakan jujur "Data spesifik tidak ditemukan di database."
- Bahasa formal dan jelas.

JAWABAN:
"""

        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2
            )

            answer = response.choices[0].message.content
            dur = time.time() - start
            return answer, dur

        except Exception as e:
            return f"Error: {e}", 0

    # ------------------------------------------------
    # DEBUG RETRIEVAL
    # ------------------------------------------------
    def debug_retrieval(self, question: str, n: int = 5):
        """Menampilkan top-N dokumen hasil re-ranking"""
        docs = self.answer_question(question, top_k=n)
        print(f"\nTop {n} Dokumen untuk Query: {question}")
        for i, doc in enumerate(docs):
            print(f"{i+1}. Skor {doc.metadata['rerank_score']:.4f} → {doc.page_content[:200]}...")



# ====================================================
# MAIN UNTUK EKSEKUSI LANGSUNG
# ====================================================
def main():
    DB = "./chroma_gempa_db"
    API_KEY = api_key

    expert = GempaExpert(DB, API_KEY)

    print("\n=== Sistem Pakar Gempa (RAG + LLM) ===")

    while True:
        print("\nMenu:")
        print("1. Tanya analisis gempa")
        print("2. Debug retrieval")
        print("3. Keluar")

        choice = input("\nPilih menu: ")

        if choice == "1":
            q = input("Pertanyaan: ")
            ans, dur = expert.generate_answer(q)
            print("\n---- HASIL ----")
            print(ans)
            print(f"\n(waktu: {dur:.2f} detik)")

        elif choice == "2":
            q = input("Masukkan query debug: ")
            expert.debug_retrieval(q)

        elif choice == "3":
            print("Selesai.")
            break

        else:
            print("Pilihan tidak valid.")


if __name__ == "__main__":
    main()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: cpu
INFO:backoff:Backing off send_request(...) for 0.1s (requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Read timed out. (read timeout=15))
INFO:GempaExpert:BM25 siap dengan 157332 dokumen.
INFO:GempaExpert:GempaExpert siap: Hybrid Retrieval + Re-ranker + LLM diaktifkan.



=== Sistem Pakar Gempa (RAG + LLM) ===

Menu:
1. Tanya analisis gempa
2. Debug retrieval
3. Keluar


Batches: 100%|██████████| 33/33 [00:20<00:00,  1.64it/s]
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



---- HASIL ----
Berdasarkan data yang tersedia, beberapa gempa bumi yang terjadi di Aceh pada tahun 2022 adalah:

1. Pada tanggal 12 June 2022 pukul 17:45 WIB, terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.7 dan kedalaman 117.11 km.
2. Pada tanggal 23 August 2022 pukul 16:45 WIB, terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.6 dan kedalaman 68.048 km.
3. Pada tanggal 22 September 2022 pukul 19:36 WIB, terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.8 dan kedalaman 9.746 km.
4. Pada tanggal 31 July 2022 pukul 03:03 WIB, terjadi gempa bumi di Provinsi Aceh dengan magnitudo 5.1 dan kedalaman 74.145 km.
5. Pada tanggal 02 September 2022 pukul 03:34 WIB, terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.7 dan kedalaman 60.247 km.
6. Pada tanggal 11 September 2022 pukul 15:35 WIB, terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4.6 dan kedalaman 90.991 km.
7. Pada tanggal 15 March 2022 pukul 09:08 WIB, terjadi gempa bumi di Provinsi Aceh dengan magnitudo 4